# DPO Fine-tuning Experiments

Учебная работа: сравнение трёх языковых моделей до и после DPO. Сохранены результаты первоначального запуска; после оформления вычисления повторно не выполнялись.

### Исходное задание

## Задание 1 (8 баллов)

Дообучите три любые разные модели с использованием DPO. Можете использовать другие датасеты с chosen/rejected структурой. Попробуйте обучать модель на как можно большем количестве примеров, которые позволит вам colab. Можете попробовать менять параметры обучения тоже.

Для каждой дообученой модели подготовьте небольшой репорт, в котором вы сравниваете исходные предсказания и предсказания после дообучения. Найдите несколько закономерностей глазами, а также посчитайте простые статистики (длина, количество уникальных символов и тп)

В конце проверьте есть ли пересечения паттернов между моделями.

## Окружение

Исходный запуск: Google Colab, Tesla T4. Используйте отдельное окружение с PyTorch (для GPU — соответствующая сборка CUDA). Ниже приведены версии из последней установочной ячейки исходного ноутбука как отправная точка; полная совместимость не проверена. Не удаляйте системные пакеты вручную.

```bash
python -m pip install numpy==1.26.4 scipy==1.13.1 transformers==4.46.3 tokenizers==0.20.3 accelerate==1.1.1 datasets==3.1.0 trl==0.12.2 peft==0.13.2 huggingface_hub==0.26.5 safetensors==0.4.5 pandas tqdm ipython
```

После установки перезапустите kernel, затем выполняйте вычислительные ячейки.

## Импорты и seed

In [4]:
import os
import re
import gc
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from IPython.display import display, Markdown
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


## Конфигурация экспериментов

In [5]:
DATASET_NAME = "Dahoas/full-hh-rlhf"

N_TRAIN = 2000
N_EVAL = 100
PREF_EVAL_N = 60

EXPERIMENTS = [
    {
        "model_name": "sshleifer/tiny-gpt2",
        "short_name": "tiny_gpt2",
        "max_steps": 80,
        "learning_rate": 1e-4,
    },
    {
        "model_name": "distilgpt2",
        "short_name": "distilgpt2",
        "max_steps": 120,
        "learning_rate": 5e-5,
    },
    {
        "model_name": "EleutherAI/pythia-70m-deduped",
        "short_name": "pythia70m",
        "max_steps": 120,
        "learning_rate": 5e-5,
    },
]

GEN_PROMPTS = [
    "\n\nHuman: Explain why the sky is blue in simple words.\n\nAssistant:",
    "\n\nHuman: I am very nervous before an exam. Give me practical advice.\n\nAssistant:",
    "\n\nHuman: Write a short polite email asking a professor for an extension.\n\nAssistant:",
    "\n\nHuman: What are three safe ways to improve sleep schedule?\n\nAssistant:",
    "\n\nHuman: Explain what machine learning is to a beginner.\n\nAssistant:",
]

OUTPUT_DIR = Path("dpo_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## Подготовка датасета

In [6]:
def strip_prompt_from_completion(prompt: str, completion: str) -> str:
    prompt = prompt or ""
    completion = completion or ""
    if completion.startswith(prompt):
        return completion[len(prompt):]
    return completion


def preprocess_example(ex):
    prompt = str(ex["prompt"])
    chosen = strip_prompt_from_completion(prompt, str(ex["chosen"]))
    rejected = strip_prompt_from_completion(prompt, str(ex["rejected"]))
    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }


def valid_example(ex):
    # Отсекаем пустые и слишком длинные примеры, чтобы обучение было стабильнее и быстрее.
    if not ex["prompt"].strip() or not ex["chosen"].strip() or not ex["rejected"].strip():
        return False
    total_len = len(ex["prompt"]) + max(len(ex["chosen"]), len(ex["rejected"]))
    return total_len < 3500

raw = load_dataset(DATASET_NAME, split="train")
print(raw)

prepared = (
    raw
    .shuffle(seed=SEED)
    .map(preprocess_example, remove_columns=raw.column_names)
    .filter(valid_example)
)

needed = N_TRAIN + N_EVAL
if len(prepared) < needed:
    raise ValueError(f"После фильтрации осталось только {len(prepared)} примеров, нужно {needed}.")

train_ds = prepared.select(range(N_TRAIN))
eval_ds = prepared.select(range(N_TRAIN, N_TRAIN + N_EVAL))

print("train:", train_ds)
print("eval:", eval_ds)
print("Пример:")
print(json.dumps(train_ds[0], ensure_ascii=False, indent=2)[:2000])


Dataset({
    features: ['prompt', 'response', 'chosen', 'rejected'],
    num_rows: 112052
})
train: Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 2000
})
eval: Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 100
})
Пример:
{
  "prompt": "\n\nHuman: what do i do if i am coughing up blood?\n\nAssistant: If you are coughing up blood, this is a sign of a more serious illness.  As with all medical situations, you should see a doctor to get advice from a medical professional.  Do you have any health insurance?\n\nHuman: No, I dont.\n\nAssistant:",
  "chosen": " That’s unfortunate, I’m afraid that without health insurance you can’t always afford a doctor.  It is often possible to obtain discounted health services in the United States, if you can prove you have very limited income, or if you are a veteran of the armed forces.  Otherwise, if you are uninsured, it is possible that your health insurance has been denied by the medical insurance co

## Генерация и метрики предпочтений

In [13]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_model_device(model):
    return next(model.parameters()).device


def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(DEVICE)
    return model, tokenizer


def generate_responses(model, tokenizer, prompts, max_new_tokens=90):
    model.eval()
    device = get_model_device(model)
    rows = []
    for i, prompt in enumerate(prompts):
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=384,
            padding=False,
        ).to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        response = decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded.strip()
        rows.append({
            "prompt_id": i,
            "prompt": prompt,
            "response": response,
        })
    return rows


def text_stats(text: str):
    text = text or ""
    words = re.findall(r"[A-Za-zА-Яа-яЁё0-9']+", text.lower())
    chars_no_space = [c for c in text if not c.isspace()]
    unique_words = set(words)
    unique_chars = set(chars_no_space)
    repetition_ratio = 1 - (len(unique_words) / max(len(words), 1))
    return {
        "chars": len(text),
        "chars_no_space": len(chars_no_space),
        "words": len(words),
        "unique_words": len(unique_words),
        "unique_chars": len(unique_chars),
        "repetition_ratio": repetition_ratio,
        "lines": text.count("\n") + 1 if text else 0,
    }


def generations_to_stats_df(generation_rows):
    rows = []
    for row in generation_rows:
        stats = text_stats(row["response"])
        rows.append({**row, **stats})
    return pd.DataFrame(rows)


def mean_completion_logprob(model, tokenizer, prompt: str, completion: str, max_length=512):
    """Средний log P(completion | prompt) по токенам completion."""
    model.eval()
    device = get_model_device(model)

    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    full_text = prompt + completion
    enc = tokenizer(
        full_text,
        add_special_tokens=False,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    ).to(device)

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]
    if input_ids.shape[1] < 2:
        return np.nan

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    # logits[:, t] предсказывает token t+1
    logits = logits[:, :-1, :]
    labels = input_ids[:, 1:]
    log_probs = torch.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)

    # Completion начинается после токенов prompt.
    prompt_len = min(len(prompt_ids), input_ids.shape[1] - 1)
    start_label_pos = max(prompt_len - 1, 0)
    completion_mask = torch.zeros_like(labels, dtype=torch.bool)
    completion_mask[:, start_label_pos:] = True
    completion_mask = completion_mask & attention_mask[:, 1:].bool()

    if completion_mask.sum().item() == 0:
        return np.nan
    return token_log_probs[completion_mask].mean().item()


def preference_metrics(model, tokenizer, dataset, n=PREF_EVAL_N, max_length=512):
    n = min(n, len(dataset))
    rows = []
    for i in tqdm(range(n), desc="preference eval"):
        ex = dataset[i]
        chosen_lp = mean_completion_logprob(model, tokenizer, ex["prompt"], ex["chosen"], max_length=max_length)
        rejected_lp = mean_completion_logprob(model, tokenizer, ex["prompt"], ex["rejected"], max_length=max_length)
        rows.append({
            "example_id": i,
            "chosen_logp": chosen_lp,
            "rejected_logp": rejected_lp,
            "margin": chosen_lp - rejected_lp,
            "pref_correct": chosen_lp > rejected_lp,
            "prompt_preview": ex["prompt"][:180].replace("\n", " "),
            "chosen_preview": ex["chosen"][:220].replace("\n", " "),
            "rejected_preview": ex["rejected"][:220].replace("\n", " "),
        })
    df = pd.DataFrame(rows).dropna(subset=["chosen_logp", "rejected_logp"])
    metrics = {
        "preference_accuracy": float(df["pref_correct"].mean()) if len(df) else np.nan,
        "mean_margin": float(df["margin"].mean()) if len(df) else np.nan,
        "median_margin": float(df["margin"].median()) if len(df) else np.nan,
        "n_eval_pairs": int(len(df)),
    }
    return metrics, df

## DPO-обучение одной модели

In [14]:
def run_dpo_experiment(config, train_ds, eval_ds):
    model_name = config["model_name"]
    short_name = config["short_name"]
    max_steps = config["max_steps"]
    learning_rate = config["learning_rate"]

    display(Markdown(f"## Эксперимент: `{model_name}`"))
    print("Загрузка модели...")
    model, tokenizer = load_model_and_tokenizer(model_name)

    print("Генерации ДО DPO...")
    base_generations = generate_responses(model, tokenizer, GEN_PROMPTS)
    base_gen_df = generations_to_stats_df(base_generations)
    base_gen_df["model"] = short_name
    base_gen_df["stage"] = "base"

    print("Preference metrics ДО DPO...")
    base_pref_metrics, base_pair_df = preference_metrics(model, tokenizer, eval_ds)
    base_pair_df["model"] = short_name
    base_pair_df["stage"] = "base"

    print("Запуск DPO training...")
    args = DPOConfig(
        output_dir=str(OUTPUT_DIR / short_name),
        max_steps=max_steps,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=learning_rate,
        warmup_ratio=0.05,
        logging_steps=10,
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
        optim="adamw_torch",
        fp16=torch.cuda.is_available(),
        bf16=False,
        gradient_checkpointing=False,
        max_length=512,
        beta=0.1,
        seed=SEED,
    )

    trainer = DPOTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
    )

    train_output = trainer.train()
    train_metrics = train_output.metrics
    trainer.save_model(str(OUTPUT_DIR / short_name / "final"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / short_name / "final"))

    tuned_model = trainer.model
    print("Генерации ПОСЛЕ DPO...")
    tuned_generations = generate_responses(tuned_model, tokenizer, GEN_PROMPTS)
    tuned_gen_df = generations_to_stats_df(tuned_generations)
    tuned_gen_df["model"] = short_name
    tuned_gen_df["stage"] = "dpo"

    print("Preference metrics ПОСЛЕ DPO...")
    tuned_pref_metrics, tuned_pair_df = preference_metrics(tuned_model, tokenizer, eval_ds)
    tuned_pair_df["model"] = short_name
    tuned_pair_df["stage"] = "dpo"

    metrics_row = {
        "model": short_name,
        "model_name": model_name,
        "n_train": len(train_ds),
        "n_eval": len(eval_ds),
        "max_steps": max_steps,
        "learning_rate": learning_rate,
        "base_preference_accuracy": base_pref_metrics["preference_accuracy"],
        "dpo_preference_accuracy": tuned_pref_metrics["preference_accuracy"],
        "delta_preference_accuracy": tuned_pref_metrics["preference_accuracy"] - base_pref_metrics["preference_accuracy"],
        "base_mean_margin": base_pref_metrics["mean_margin"],
        "dpo_mean_margin": tuned_pref_metrics["mean_margin"],
        "delta_mean_margin": tuned_pref_metrics["mean_margin"] - base_pref_metrics["mean_margin"],
        **{f"train_{k}": v for k, v in train_metrics.items()},
    }

    gen_df = pd.concat([base_gen_df, tuned_gen_df], ignore_index=True)
    pair_df = pd.concat([base_pair_df, tuned_pair_df], ignore_index=True)

    # Сохраняем промежуточные CSV, чтобы не потерять результаты.
    gen_df.to_csv(OUTPUT_DIR / short_name / "generations_before_after.csv", index=False)
    pair_df.to_csv(OUTPUT_DIR / short_name / "preference_eval_pairs.csv", index=False)
    pd.DataFrame([metrics_row]).to_csv(OUTPUT_DIR / short_name / "metrics_summary.csv", index=False)

    display(Markdown(f"### Краткий итог `{short_name}`"))
    display(pd.DataFrame([metrics_row]))
    display(gen_df[["model", "stage", "prompt_id", "response", "words", "unique_words", "repetition_ratio"]])

    # Чистим память перед следующей моделью.
    del trainer, tuned_model, model
    clear_memory()

    return metrics_row, gen_df, pair_df


## Запуск трёх экспериментов

In [15]:
all_metric_rows = []
all_generation_dfs = []
all_pair_dfs = []

for cfg in EXPERIMENTS:
    metrics_row, gen_df, pair_df = run_dpo_experiment(cfg, train_ds, eval_ds)
    all_metric_rows.append(metrics_row)
    all_generation_dfs.append(gen_df)
    all_pair_dfs.append(pair_df)

metrics_df = pd.DataFrame(all_metric_rows)
generations_df = pd.concat(all_generation_dfs, ignore_index=True)
pairs_df = pd.concat(all_pair_dfs, ignore_index=True)

metrics_df.to_csv(OUTPUT_DIR / "all_metrics_summary.csv", index=False)
generations_df.to_csv(OUTPUT_DIR / "all_generations_before_after.csv", index=False)
pairs_df.to_csv(OUTPUT_DIR / "all_preference_eval_pairs.csv", index=False)

print("Готово. Все результаты сохранены в папку:", OUTPUT_DIR.resolve())

## Эксперимент: `sshleifer/tiny-gpt2`

Загрузка модели...
Генерации ДО DPO...
Preference metrics ДО DPO...


Запуск DPO training...


Step,Training Loss
10,0.693100
20,0.693000
30,0.693200
40,0.693100
50,0.692900
60,0.691500
70,0.691400
80,0.691600


Генерации ПОСЛЕ DPO...
Preference metrics ПОСЛЕ DPO...


### Краткий итог `tiny_gpt2`

... (17 outputs left out)

### Краткий итог `pythia70m`

,model,model_name,n_train,n_eval,max_steps,learning_rate,base_preference_accuracy,dpo_preference_accuracy,delta_preference_accuracy,base_mean_margin,dpo_mean_margin,delta_mean_margin,train_train_runtime,train_train_samples_per_second,train_train_steps_per_second,train_total_flos,train_train_loss,train_epoch
0,pythia70m,EleutherAI/pythia-70m-deduped,2000,100,120,0.00005,0.466667,0.483333,0.016667,-0.140461,-0.134548,0.005913,81.7508,23.486,1.468,0.0,2.190415,0.96


,model,stage,prompt_id,response,words,unique_words,repetition_ratio
0,pythia70m,base,0,I believe the world is very different from the...,66,15,0.772727
1,pythia70m,base,1,1. You have no idea how much to ask for.\n\n2....,63,29,0.539683
2,pythia70m,base,2,Marcelo Pérez-\n\nGómez\n\nGómez-\n\nGómez-\n\...,27,6,0.777778
3,pythia70m,base,3,The answer to the question is not about the nu...,72,49,0.319444
4,pythia70m,base,4,Develop an online learning model for a new too...,12,12,0.000000
5,pythia70m,dpo,0,Avenger\n\n1. The main point of the Earth is b...,72,16,0.777778
6,pythia70m,dpo,1,Exposure to air loss and exposure to air loss ...,75,48,0.360000
7,pythia70m,dpo,2,The New England Journal of the National Scienc...,64,21,0.671875
8,pythia70m,dpo,3,Sleep Medicine (Sociodemus)\n\nSociodemus\n\nS...,17,4,0.764706
9,pythia70m,dpo,4,1. 2. 2. 2. 1. 2. 3. 4. 5. 6. 6. 6....,38,6,0.842105


Готово. Все результаты сохранены в папку: /content/dpo_outputs


## Сравнение метрик и генераций

In [16]:

display(Markdown("### Метрики preference accuracy и margin"))
display(metrics_df[[
    "model", "model_name", "n_train", "max_steps", "learning_rate",
    "base_preference_accuracy", "dpo_preference_accuracy", "delta_preference_accuracy",
    "base_mean_margin", "dpo_mean_margin", "delta_mean_margin",
]])

display(Markdown("### Средние статистики генераций до/после"))
gen_summary = (
    generations_df
    .groupby(["model", "stage"], as_index=False)
    [["chars", "words", "unique_words", "unique_chars", "repetition_ratio"]]
    .mean()
)
display(gen_summary)

display(Markdown("### Все генерации до/после"))
display(generations_df[["model", "stage", "prompt_id", "prompt", "response", "words", "unique_words", "repetition_ratio"]])


### Метрики preference accuracy и margin

,model,model_name,n_train,max_steps,learning_rate,base_preference_accuracy,dpo_preference_accuracy,delta_preference_accuracy,base_mean_margin,dpo_mean_margin,delta_mean_margin
0,tiny_gpt2,sshleifer/tiny-gpt2,2000,80,0.00010,0.583333,0.483333,-0.100000,0.000044,0.000144,0.000100
1,distilgpt2,distilgpt2,2000,120,0.00005,0.566667,0.516667,-0.050000,-0.040802,-0.013195,0.027607
2,pythia70m,EleutherAI/pythia-70m-deduped,2000,120,0.00005,0.466667,0.483333,0.016667,-0.140461,-0.134548,0.005913


### Средние статистики генераций до/после

,model,stage,chars,words,unique_words,unique_chars,repetition_ratio
0,distilgpt2,base,408.8,74.0,12.8,22.6,0.823807
1,distilgpt2,dpo,389.2,67.2,16.0,23.6,0.753643
2,pythia70m,base,260.8,48.0,22.2,24.8,0.481926
3,pythia70m,dpo,301.4,53.2,19.0,22.8,0.683293
4,tiny_gpt2,base,616.6,63.4,43.2,34.0,0.318528
5,tiny_gpt2,dpo,638.2,67.0,44.6,32.6,0.334173


### Все генерации до/после

,model,stage,prompt_id,prompt,response,words,unique_words,repetition_ratio
0,tiny_gpt2,base,0,\n\nHuman: Explain why the sky is blue in simp...,mediatelyJDatisfRocket ESV intermittentRocket ...,64,44,0.312500
1,tiny_gpt2,base,1,\n\nHuman: I am very nervous before an exam. G...,Observ trilogy confiratisf Motorola vendors Mo...,65,46,0.292308
2,tiny_gpt2,base,2,\n\nHuman: Write a short polite email asking a...,HancockScenepress hauled conservationpressmedi...,60,39,0.350000
3,tiny_gpt2,base,3,\n\nHuman: What are three safe ways to improve...,Participation vendors Motorola autonomy interm...,62,45,0.274194
4,tiny_gpt2,base,4,\n\nHuman: Explain what machine learning is to...,hauled conservationJD Observ trilogy substJDpr...,66,42,0.363636
5,tiny_gpt2,dpo,0,\n\nHuman: Explain why the sky is blue in simp...,mediatelyJD RhRocket ESV intermittentRocket Ha...,66,45,0.318182
6,tiny_gpt2,dpo,1,\n\nHuman: I am very nervous before an exam. G...,Observ possibility confiriken Motorola respons...,70,45,0.357143
7,tiny_gpt2,dpo,2,\n\nHuman: Write a short polite email asking a...,HancockScenepress hauled conservationpressmedi...,63,41,0.349206
8,tiny_gpt2,dpo,3,\n\nHuman: What are three safe ways to improve...,Participation vendors Motorola autonomy respon...,67,47,0.298507
9,tiny_gpt2,dpo,4,\n\nHuman: Explain what machine learning is to...,hauled conservation possibility Observ trilogy...,69,45,0.347826


## Отчёты по моделям

In [17]:

def signed_word(delta, positive="увеличилось", negative="уменьшилось"):
    if pd.isna(delta):
        return "не посчитано"
    if delta > 0:
        return positive
    if delta < 0:
        return negative
    return "не изменилось"


def get_gen_delta(model, metric):
    sub = gen_summary[gen_summary["model"] == model].set_index("stage")
    if "base" not in sub.index or "dpo" not in sub.index:
        return np.nan
    return float(sub.loc["dpo", metric] - sub.loc["base", metric])


def best_example_for_model(model):
    sub = generations_df[generations_df["model"] == model]
    rows = []
    for pid in sorted(sub["prompt_id"].unique()):
        base = sub[(sub["prompt_id"] == pid) & (sub["stage"] == "base")]
        dpo = sub[(sub["prompt_id"] == pid) & (sub["stage"] == "dpo")]
        if len(base) and len(dpo):
            base = base.iloc[0]
            dpo = dpo.iloc[0]
            score = (dpo["words"] - base["words"]) + 2 * (dpo["unique_words"] - base["unique_words"]) - 10 * (dpo["repetition_ratio"] - base["repetition_ratio"])
            rows.append((score, pid, base, dpo))
    if not rows:
        return None
    rows.sort(key=lambda x: x[0], reverse=True)
    return rows[0]


for _, m in metrics_df.iterrows():
    model = m["model"]
    words_delta = get_gen_delta(model, "words")
    unique_delta = get_gen_delta(model, "unique_words")
    repetition_delta = get_gen_delta(model, "repetition_ratio")

    report = []
    report.append(f"## Репорт: `{model}`")
    report.append(f"**Модель:** `{m['model_name']}`")
    report.append("")
    report.append("### 1. Preference accuracy")
    report.append(
        f"До DPO модель предпочитала `chosen` в **{m['base_preference_accuracy']:.3f}** случаев; "
        f"после DPO — в **{m['dpo_preference_accuracy']:.3f}** случаев. "
        f"Изменение: **{m['delta_preference_accuracy']:+.3f}**."
    )
    report.append(
        f"Средний margin `logP(chosen)-logP(rejected)` изменился с **{m['base_mean_margin']:.4f}** "
        f"до **{m['dpo_mean_margin']:.4f}**. Изменение: **{m['delta_mean_margin']:+.4f}**."
    )
    report.append("")
    report.append("### 2. Простые статистики генераций")
    report.append(f"Средняя длина в словах {signed_word(words_delta)} на **{words_delta:+.2f}**.")
    report.append(f"Среднее число уникальных слов {signed_word(unique_delta)} на **{unique_delta:+.2f}**.")
    report.append(f"Доля повторов {signed_word(repetition_delta)} на **{repetition_delta:+.3f}**.")
    report.append("")
    report.append("### 3. Пример до/после")
    ex = best_example_for_model(model)
    if ex is not None:
        _, pid, base, dpo = ex
        report.append(f"**Prompt {pid}:** `{base['prompt']}`")
        report.append("")
        report.append("**До DPO:**")
        report.append(str(base["response"])[:1200])
        report.append("")
        report.append("**После DPO:**")
        report.append(str(dpo["response"])[:1200])
    display(Markdown("\n".join(report)))


## Репорт: `tiny_gpt2`
**Модель:** `sshleifer/tiny-gpt2`

### 1. Preference accuracy
До DPO модель предпочитала `chosen` в **0.583** случаев; после DPO — в **0.483** случаев. Изменение: **-0.100**.
Средний margin `logP(chosen)-logP(rejected)` изменился с **0.0000** до **0.0001**. Изменение: **+0.0001**.

### 2. Простые статистики генераций
Средняя длина в словах увеличилось на **+3.60**.
Среднее число уникальных слов увеличилось на **+1.40**.
Доля повторов увеличилось на **+0.016**.

### 3. Пример до/после
**Prompt 4:** `

Human: Explain what machine learning is to a beginner.

Assistant:`

**До DPO:**
hauled conservationJD Observ trilogy substJDpress credibility vendors subst autonomy Money TAatisf conservation Moneypress Rh Prob conservation ProbRocketpress circumcised confir Habit reviewing trilogy ONE reviewing confiroho conservation antibioticpressScene confir Participation TAoho Brew Motorola intermittent ONERocket Jr Brewpress ONE pawnootherikenhibitScene directly Motorola scalp reviewing scalp Motorola stairs Jr TAJD pawn Prob pawnatisfRocketatisfoother credibility stairsoho Observ directly Motorolaiken dispatch reviewing Habit ONE subst Daniel pawn intermittent scalp vendors hauled

**После DPO:**
hauled conservation possibility Observ trilogy substJDpress credibility vendors subst autonomy Money TA things possibility Moneypress Rh Prob conservation ProbRocketpress circumcised confir Habit reviewing trilogyJD reviewing confiroho conservation antibioticpressScene responsibly Participation TAoho Brew Motorola intermittent scalpRocket Jr Brewpress Daniel pawn substikenhibitScene directly Motorola scalp reviewing scalp Motorola stairs Jr TAJD pawn Prob pawn confirRocketting TA credibility stairsoho Observ directly Motorolaiken dispatch possibility Habitting subst Daniel pawn intermittent scalp vendors hauled

## Репорт: `distilgpt2`
**Модель:** `distilgpt2`

### 1. Preference accuracy
До DPO модель предпочитала `chosen` в **0.567** случаев; после DPO — в **0.517** случаев. Изменение: **-0.050**.
Средний margin `logP(chosen)-logP(rejected)` изменился с **-0.0408** до **-0.0132**. Изменение: **+0.0276**.

### 2. Простые статистики генераций
Средняя длина в словах уменьшилось на **-6.80**.
Среднее число уникальных слов увеличилось на **+3.20**.
Доля повторов уменьшилось на **-0.070**.

### 3. Пример до/после
**Prompt 3:** `

Human: What are three safe ways to improve sleep schedule?

Assistant:`

**До DPO:**
The right way to improve sleep schedule is by using the right way to increase sleep time by using the right way to increase sleep time by using the right way to increase sleep time by using the right way to increase sleep time by using the right way to increase sleep time by using the right way to increase sleep time by using the right way to increase sleep time by using the right way to increase sleep time by using the right way to increase sleep time by

**После DPO:**
Sleep is a process that involves changing your circadian rhythm and slowing your activity. In addition, you can make a change in your sleep schedule and make it easier for you to get up and down, and to get to work. The more you work, the more time you'll be able to get up and down, and to get to work.

## Репорт: `pythia70m`
**Модель:** `EleutherAI/pythia-70m-deduped`

### 1. Preference accuracy
До DPO модель предпочитала `chosen` в **0.467** случаев; после DPO — в **0.483** случаев. Изменение: **+0.017**.
Средний margin `logP(chosen)-logP(rejected)` изменился с **-0.1405** до **-0.1345**. Изменение: **+0.0059**.

### 2. Простые статистики генераций
Средняя длина в словах увеличилось на **+5.20**.
Среднее число уникальных слов уменьшилось на **-3.20**.
Доля повторов увеличилось на **+0.201**.

### 3. Пример до/после
**Prompt 2:** `

Human: Write a short polite email asking a professor for an extension.

Assistant:`

**До DPO:**
Marcelo Pérez-

Gómez

Gómez-

Gómez-

Gómez-

Gómez-

Gómez-

Gómez-

Gómez-

Gómez-

Gómez-

Gómez-

Góme

**После DPO:**
The New England Journal of the National Science and Technology Policy and Technology (NIST) in New England (Westminster University, England)

Institution of the Ministry of Science and Space for Science and Space (SIRIS) in New England, New England, New England, New England University of New England (WNAS) and New England University of New England (WNAS).  The National Science Foundation of the National Science Foundation

## Общие закономерности

In [18]:
patterns = []
for model in metrics_df["model"]:
    row = metrics_df[metrics_df["model"] == model].iloc[0]
    patterns.append({
        "model": model,
        "pref_accuracy_improved": row["delta_preference_accuracy"] > 0,
        "margin_improved": row["delta_mean_margin"] > 0,
        "answers_longer": get_gen_delta(model, "words") > 0,
        "more_unique_words": get_gen_delta(model, "unique_words") > 0,
        "less_repetition": get_gen_delta(model, "repetition_ratio") < 0,
    })

patterns_df = pd.DataFrame(patterns)
display(patterns_df)

common_patterns = []
for col in ["pref_accuracy_improved", "margin_improved", "answers_longer", "more_unique_words", "less_repetition"]:
    count = int(patterns_df[col].sum())
    if count == len(patterns_df):
        common_patterns.append(f"Паттерн `{col}` есть у всех 3 моделей.")
    elif count >= 2:
        common_patterns.append(f"Паттерн `{col}` есть у {count} из 3 моделей.")
    else:
        common_patterns.append(f"Паттерн `{col}` не является общим: есть только у {count} из 3 моделей.")

display(Markdown("## Общие закономерности\n" + "\n".join([f"- {p}" for p in common_patterns])))


,model,pref_accuracy_improved,margin_improved,answers_longer,more_unique_words,less_repetition
0,tiny_gpt2,False,True,True,True,False
1,distilgpt2,False,True,False,True,True
2,pythia70m,True,True,True,False,False


## Общие закономерности
- Паттерн `pref_accuracy_improved` не является общим: есть только у 1 из 3 моделей.
- Паттерн `margin_improved` есть у всех 3 моделей.
- Паттерн `answers_longer` есть у 2 из 3 моделей.
- Паттерн `more_unique_words` есть у 2 из 3 моделей.
- Паттерн `less_repetition` не является общим: есть только у 1 из 3 моделей.

## Интерпретация и ограничения

Средний margin вырос у всех моделей, но preference accuracy снизилась у Tiny GPT-2 и DistilGPT-2; у Pythia-70M прирост составил один пример из 60. Длина и доля повторов не измеряют качество ответа напрямую. Функция оценки чувствительна к обрезанию последовательностей и границе prompt/completion. Повторные запуски и независимая оценка качества не проводились.

Дополнительное задание исходной работы требовало доказать улучшение качества моделей. Имеющихся результатов недостаточно для такого вывода; автоматическая аргументация удалена при оформлении.